# Hybrid meta-cognition agent

This notebook demonstrates the reusable hybrid-agent loop in a more explanatory, step-by-step way.

## What this notebook shows
- discover reusable steering controllers from feature specifications
- load those controllers into the agent's memory
- route a task through planning, retrieval, tool use, execution, and verification
- inspect the resulting controller choice and the learned interaction traces

The notebook keeps the demo wiring local, while the shared `activation_steering` package supplies the reusable implementation.


## Step 1: initialize the demo

We pick one instruction-tuned model, a single task, and a temporary artifact file that will hold the discovered controllers.

The task asks for both an answer and supporting evidence so the planner has a clear reason to request retrieval.


In [ ]:
from pathlib import Path

from transformers import set_seed

import activation_steering as steering

set_seed(0)

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
LAYER_IDX = 12
TASK = "What is the capital of France, and what evidence supports that answer?"
ARTIFACT_PATH = Path("/tmp/hybrid_agent_feature_vectors.json")

model, tokenizer = steering.load_model_and_tokenizer(MODEL_NAME)
device = steering.get_model_device(model)
resolved_layer_idx = min(LAYER_IDX, len(steering.get_transformer_layers(model)) - 1)

print(f"Model: {MODEL_NAME}")
print(f"Resolved layer index: {resolved_layer_idx}")
print(f"Artifact path: {ARTIFACT_PATH}")


## Step 2: discover and load steering controllers

The repository ships with reusable feature specifications. Here we convert those specs into steering vectors for the current model, persist them, and reload them as named controllers.

The memory object then becomes the agent's working catalog of available steering policies.


In [ ]:
feature_specs = steering.get_standard_feature_specs()
discovered_vectors = steering.discover_and_store_feature_vectors(
    feature_specs=feature_specs,
    layer_idx=resolved_layer_idx,
    model=model,
    tokenizer=tokenizer,
    device=device,
    output_path=ARTIFACT_PATH,
)

controllers = steering.load_steering_controllers(
    ARTIFACT_PATH,
    default_alpha=1.0,
    default_decay=0.9,
    task_types_by_controller={
        "retrieval_augmented_context": ["qa"],
        "chain_of_thought": ["qa"],
    },
)

memory = steering.InMemorySteeringMemory(controllers)
executor = steering.SteeredExecutor(
    model=model,
    tokenizer=tokenizer,
    device=device,
    model_name=MODEL_NAME,
)

print(f"Persisted {len(discovered_vectors)} feature vectors to {ARTIFACT_PATH}")
print("Controllers:", [controller.controller_id for controller in controllers])


## Step 3: define the notebook-local agent components

Each component focuses on one job:

- the **planner** decides whether retrieval is needed and which controller to request
- the **retriever** returns evidence snippets
- the **tool router** can add extra observations from external tools
- the **verifier** checks whether the draft satisfies the notebook's grounding requirement

In production code these pieces could come from separate systems, but the notebook keeps them explicit so the control flow is easy to inspect.


In [ ]:
def planner(task: str, memory_store: steering.InMemorySteeringMemory) -> steering.PlannerDecision:
    needs_retrieval = "evidence" in task.lower() or "support" in task.lower()
    return steering.PlannerDecision(
        task_type="qa",
        needs_retrieval=needs_retrieval,
        controller_id="retrieval_augmented_context" if needs_retrieval else "chain_of_thought",
        reasoning_effort="medium",
        use_steering=True,
        allow_fallback=True,
    )


def retriever(task: str, plan: steering.PlannerDecision) -> list[str]:
    return [
        "Reference: Paris is the capital and most populous city of France.",
        "Reference: France is a country in Western Europe.",
    ]


def tool_router(task: str, plan: steering.PlannerDecision, context: list[str]) -> list[str]:
    return ["Tool observation: retrieved references are sufficient for a grounded short answer."]


def verifier(
    task: str,
    draft: steering.ExecutorResult,
    context: list[str],
    plan: steering.PlannerDecision,
) -> steering.VerifierResult:
    has_context = bool(context)
    mentions_paris = "paris" in draft.output_text.lower()
    return steering.VerifierResult(
        passed=has_context and mentions_paris,
        confidence=0.8 if has_context and mentions_paris else 0.3,
        issues=[] if has_context and mentions_paris else ["missing grounded evidence"],
    )


## Step 4: run the agent and inspect the result

A single call to `HybridMetaCognitionAgent.run(...)` ties together planning, retrieval, optional steering, verification, and memory write-back.

After the run we inspect both the output and the lightweight activation-trace metadata that the agent stores for later reflection.


In [ ]:
agent = steering.HybridMetaCognitionAgent(
    planner=planner,
    retriever=retriever,
    tool_router=tool_router,
    executor=executor,
    verifier=verifier,
    memory=memory,
    max_new_tokens=40,
)

run = agent.run(TASK)

print("Selected controller:", run.selected_controller_id)
print("Fallback used:", run.fallback_used)
print("Verifier confidence:", run.verdict.confidence)
print("Observed feature ids:", run.draft.activation_trace.observed_feature_ids)
print(
    "Top trace features:",
    [(score.feature_id, round(score.score, 3)) for score in run.draft.activation_trace.top_feature_scores],
)
print()
print("Output:")
print(run.draft.output_text)


## Step 5: inspect what the memory learned

The in-memory store keeps both persistent controllers and newly observed interaction patterns. This is the bridge between a one-off run and an agent that can later adapt based on prior traces.


In [ ]:
print("Controller stats:", memory.controller_stats())
print(
    "Dynamic features:",
    [
        {
            "feature_id": feature.feature_id,
            "summary": feature.summary,
            "observations": feature.observation_count,
        }
        for feature in memory.list_dynamic_features(MODEL_NAME)
    ],
)
